# CDM scientific comparison

## Context and methods
Compare the current product with corrected reference B, which is generated from
frozen C source without importing C or ITAMAE products. B independently changes
the gravitational constant and inverts the NFW mass function using the
principal Lambert-W branch at 50-digit precision. Full one-factor records and
A/B regeneration live in `sashimi-family/validation/references/sashimi-c`.

Run from the C repository root with the reviewed candidate wheels installed.
`tests/references` is a validation input, not a runtime dependency.

### Assumptions
The Planck-calibrated background, Yang accretion, concentration scatter,
`pert2_shanks`, `ct_th=0` and all grid values are fixed by the sidecar. Agreement
validates migration under this specification; it does not establish simulation
calibration or convergence of every observable.

In [ ]:
import hashlib, json
from pathlib import Path
import numpy as np
from sashimi_c import SubhaloProperties, diagnose_stripping_approximation
reference_path = Path('tests/references/B-all.npz')
record = json.loads(reference_path.with_suffix('.json').read_text())
assert hashlib.sha256(reference_path.read_bytes()).hexdigest() == record['artifact_sha256']
assert record['role'] == 'B'
parameters = record['calculation']['parameters']
print('B source:', record['source_revision'])
print('parameters:', parameters)

## Results: all catalog columns
The acceptance tolerance is 5e-12 for this independent B/C calculation. It
covers roundoff, interpolation evaluation and the independent inverse solver;
it is tighter than the preserved historical full-observable regression.

In [ ]:
model = SubhaloProperties()
actual = model.subhalo_properties_calc(**parameters)
with np.load(reference_path) as reference:
    differences = {}
    for index, values in enumerate(actual):
        target = reference[f'tuple_{index}']
        if values.dtype.kind == 'b':
            np.testing.assert_array_equal(values, target)
            differences[f'tuple_{index}'] = int(np.count_nonzero(values != target))
        else:
            np.testing.assert_allclose(values, target, rtol=5e-12, atol=0.)
            differences[f'tuple_{index}'] = float(np.max(abs(values-target)/np.maximum(abs(target), 1e-300)))
print(differences)
print(dict(model.catalog.metadata))

## Solver diagnostic
The perturbative default is compared to direct ODE integration. This reports
an approximation error; it does not automatically select Picard or ODE as a new
default. Dedicated grid/solver convergence artifacts are reviewed separately.

In [ ]:
diagnostic = diagnose_stripping_approximation(1e12, np.logspace(6., 10., 9), accretion_redshift=1.)
print(dict(diagnostic.summary()))
assert np.all(np.isfinite(diagnostic.relative_difference))

## Takeaways
The executed cells test every B/C catalog entry and report the direct ODE
comparison under fixed conditions. Historical fixture provenance remains
unchanged. Full convergence and prompt-cusp input limitations are tracked in
the release review record; a successful regression is not a physical calibration.